# MNIST Experiment: JimmyNAS vs VanillaNAS Comparison

Visualization of all metrics comparing JimmyNAS_I_fullyCNN vs vanillaNAS_dense

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 10)

In [ ]:
# Load data
df = pd.read_csv('../Experiments_Compare_Model/MNIST/model_log.csv')
print(df)

In [ ]:
# Prepare data for comparison
metrics = [
    ('best_acc', 'Best Accuracy'),
    ('tflite_acc', 'TFLite Accuracy'),
    ('macs', 'MACs'),
    ('flash', 'Flash (bytes)'),
    ('peak_ram', 'Peak RAM (bytes)'),
    ('param_count', 'Parameters')
]

x_labels = df['decision_variable'].values

In [ ]:
# Create individual plots for each metric with summary
for metric, title in metrics:
    vanilla_col = f'vanillaNAS_dense_{metric}'
    jimmy_col = f'JimmyNAS_I_fullyCNN_{metric}'
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    x = np.arange(len(x_labels))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, df[vanilla_col], width, label='VanillaNAS', alpha=0.8, color='#2E86AB')
    bars2 = ax.bar(x + width/2, df[jimmy_col], width, label='JimmyNAS', alpha=0.8, color='#A23B72')
    
    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.0f}' if height > 1 else f'{height:.3f}',
                ha='center', va='bottom', fontsize=8)
    
    for bar in bars2:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.0f}' if height > 1 else f'{height:.3f}',
                ha='center', va='bottom', fontsize=8)
    
    ax.set_xlabel('Decision Variable', fontsize=12, fontweight='bold')
    ax.set_ylabel(title, fontsize=12, fontweight='bold')
    ax.set_title(f'{title}: JimmyNAS vs VanillaNAS', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=45, ha='right')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add summary statistics
    vanilla_mean = df[vanilla_col].mean()
    jimmy_mean = df[jimmy_col].mean()
    diff_pct = ((jimmy_mean - vanilla_mean) / vanilla_mean) * 100
    
    summary_text = f'Avg - VanillaNAS: {vanilla_mean:.2f} | JimmyNAS: {jimmy_mean:.2f} | Diff: {diff_pct:+.2f}%'
    ax.text(0.5, 0.98, summary_text, transform=ax.transAxes,
            ha='center', va='top', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.savefig(f'mnist_{metric}_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f'\n{title}:')
    print(f'  VanillaNAS - Mean: {vanilla_mean:.4f}, Min: {df[vanilla_col].min():.4f}, Max: {df[vanilla_col].max():.4f}')
    print(f'  JimmyNAS   - Mean: {jimmy_mean:.4f}, Min: {df[jimmy_col].min():.4f}, Max: {df[jimmy_col].max():.4f}')
    print(f'  Difference: {diff_pct:+.2f}%')
    print('-' * 80)

In [ ]:
output_file = "summary_statistics_MNIST.txt"

with open(output_file, "w") as f:
    f.write("=" * 60 + "\n")
    f.write("SUMMARY STATISTICS\n")
    f.write("=" * 60 + "\n")

    for metric, title in metrics:
        vanilla_col = f"vanillaNAS_dense_{metric}"
        jimmy_col = f"JimmyNAS_I_fullyCNN_{metric}"

        vanilla_mean = df[vanilla_col].mean()
        jimmy_mean = df[jimmy_col].mean()
        diff = ((jimmy_mean - vanilla_mean) / vanilla_mean) * 100

        f.write(f"\n{title}:\n")
        f.write(f"  VanillaNAS avg: {vanilla_mean:.2f}\n")
        f.write(f"  JimmyNAS avg:   {jimmy_mean:.2f}\n")
        f.write(f"  Difference:     {diff:+.2f}%\n")

print(f"Summary written to {output_file}")
